## **1. Construct the Dependency Network for Workers**

1.1 Load mobility data + building data

In [8]:
import pandas as pd
import geopandas as gpd

geojson_path = '/content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/newshape_canary_wharf_buildings_classified.geojson'
buildings_gdf = gpd.read_file(geojson_path)

csv_path = '/content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/canarywharf_r10_2025-03_04.csv'
mbility_df = pd.read_csv(csv_path)
mbility_stops_df = mbility_df[mbility_df['type'] == 'stop']
mbility_stops_df.shape

(1876684, 7)

In [9]:
# Workers in CW
workers_path = '/content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/user_ids_visitortype/likely_workers_ids.csv'
workers_df = pd.read_csv(workers_path)
# display(workers_df.head())

workers_ids = workers_df['user_id'].unique()
mbility_stops_workers_df = mbility_stops_df[mbility_stops_df['id'].isin(workers_ids)]
mbility_stops_workers_df.shape

(753881, 7)

In [10]:
buildings_gdf.columns

Index(['id', 'building_name', 'building_class', 'building_subtype', 'height',
       'num_floors', 'Work', 'Food_Drink', 'Retail_Leisure', 'Education',
       'Medical', 'Transportation', 'Other_POI', 'all_poi_names', 'poi_sum',
       'dominant_share_pct', 'Retail_Food_Ratio', 'final_functional_class',
       'dominant_share_pct_formatted', 'Retail_Food_Ratio_formatted',
       'poi_names_list', 'geometry'],
      dtype='object')

1.2 Construct dependency network (spatial mapping -> extract transitions -> build Workers Dependency Network)

In [11]:
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

def build_dependency_network(mobility_df_type, buildings_gdf):
    # 1. Spatial mapping to buildings (with a 10m buffer)
    print("Step 1: Spatial mapping mobility data to buildings...")

    # Convert mobility_df_type to GeoDataFrame
    mobility_gdf = gpd.GeoDataFrame(
        mobility_df_type,
        geometry=gpd.points_from_xy(mobility_df_type.lon, mobility_df_type.lat),
        crs="EPSG:4326" # Assuming WGS84 for lat/lon
    )

    # Reproject buildings_gdf to a projected CRS for accurate buffering (e.g., British National Grid, EPSG:27700)
    buildings_proj = buildings_gdf.to_crs(epsg=27700)
    # Apply a 10m buffer
    buildings_buffered = buildings_proj.geometry.buffer(10)
    buildings_buffered_gdf = gpd.GeoDataFrame(buildings_proj[['id', 'building_name']], geometry=buildings_buffered, crs="EPSG:27700")

    # Reproject mobility data to the same CRS for spatial join
    mobility_gdf_proj = mobility_gdf.to_crs(epsg=27700)

    # Perform spatial join
    # We use 'sjoin_nearest' to find the closest building within the buffer distance.
    # This is more robust than just 'sjoin' for points possibly just outside.
    # For this task, a simple `sjoin` with `intersects` after buffering the buildings should work.
    mobility_with_buildings = gpd.sjoin(
        mobility_gdf_proj,
        buildings_buffered_gdf[['id', 'building_name', 'geometry']].rename(columns={'id': 'building_id', 'building_name': 'building_name_mapped'}),
        how="inner",
        predicate='intersects'
    )

    # Report map后的保留mobility数据比例
    retained_ratio = len(mobility_with_buildings) / len(mobility_df_type)
    print(f"  Retained mobility data after mapping to buildings: {retained_ratio:.2%}")

    # 2. Timestamp preprocessing and 3. Extract transition sequences
    print("Step 2 & 3: Preprocessing timestamps and extracting transition sequences...")

    # Sort by user ID and timestamp
    mobility_with_buildings = mobility_with_buildings.sort_values(by=['id', 'ts'])

    # Extract next building ID and next timestamp using shift
    mobility_with_buildings['next_building_id'] = mobility_with_buildings.groupby('id')['building_id'].shift(-1)
    mobility_with_buildings['next_ts'] = mobility_with_buildings.groupby('id')['ts'].shift(-1)

    # Filter out the last record for each user (where next_building_id is NaN)
    transitions_df = mobility_with_buildings.dropna(subset=['next_building_id']).copy()
    transitions_df['next_building_id'] = transitions_df['next_building_id'].astype(str) # Ensure consistent type

    # Calculate time difference in hours
    transitions_df['time_diff_hours'] = (transitions_df['next_ts'] - transitions_df['ts']) / 3600 # Convert seconds to hours

    # 3.5 过滤原地停留 (Self-loops) 与 极短时间抖动
    # 仅保留真正发生了建筑间位移的数据
    filtered_transitions = transitions_df[
        (transitions_df['building_id'] != transitions_df['next_building_id']) &
        (transitions_df['time_diff_hours'] > 0.016) # 比如过滤掉少于 1 分钟的转移
    ].copy()

    # 4. Filter and report
    print("Step 4: Filtering transitions and reporting...")

    # Filter for transitions within 24 hours
    valid_time_transitions = transitions_df[transitions_df['time_diff_hours'] <= 6]

    # Filter out buildings with less than 20 visits (as an origin building in transitions)
    building_visit_counts = valid_time_transitions['building_id'].value_counts()
    active_buildings = building_visit_counts[building_visit_counts >= 20].index

    filtered_transitions = valid_time_transitions[valid_time_transitions['building_id'].isin(active_buildings) & \
                                                  valid_time_transitions['next_building_id'].isin(active_buildings)]

    active_nodes_count = len(active_buildings)
    valid_cross_transitions_count = len(filtered_transitions)

    print(f"  Number of active buildings (nodes) with >= 20 origin visits: {active_nodes_count}")
    print(f"  Number of valid cross-building transitions (within 6h & active buildings): {valid_cross_transitions_count}")

    if filtered_transitions.empty:
        print("  No valid transitions to build a network.")
        return None

    # 5. Network construction
    print("Step 5: Constructing the dependency network...")

    # Calculate transition probabilities P(B|A)
    transition_counts = filtered_transitions.groupby(['building_id', 'next_building_id']).size().reset_index(name='count')
    origin_counts = filtered_transitions['building_id'].value_counts().reset_index(name='total_count')
    origin_counts.rename(columns={'index': 'building_id'}, inplace=True)

    transition_probs = pd.merge(transition_counts, origin_counts, on='building_id')
    transition_probs['probability'] = transition_probs['count'] / transition_probs['total_count']

    # Create a directed graph
    G = nx.DiGraph()

    # Add nodes (active buildings)
    for building_id in active_buildings:
        G.add_node(building_id)

    # Add edges with dependency strength as weight
    for _, row in transition_probs.iterrows():
        G.add_edge(row['building_id'], row['next_building_id'], weight=row['probability'], transitions=row['count'])

    # Calculate inflow and outflow for each node
    in_degree = dict(G.in_degree(weight='transitions')) # Use 'transitions' for raw count inflow
    out_degree = dict(G.out_degree(weight='transitions')) # Use 'transitions' for raw count outflow

    nx.set_node_attributes(G, in_degree, 'inflow')
    nx.set_node_attributes(G, out_degree, 'outflow')

    # Report network characteristics
    num_edges_strong_dependency = sum(1 for u, v, d in G.edges(data=True) if d['weight'] > 0.5) # Example threshold
    print(f"  Network created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    print(f"  Number of strong dependency paths (e.g., probability > 0.5): {num_edges_strong_dependency}")

    return G

In [12]:
print("Processing Workers...")
workers_dependency_network = build_dependency_network(mbility_stops_workers_df, buildings_gdf)

if workers_dependency_network:
    print("\nWorkers Dependency Network built successfully!")
    print(f"Nodes: {workers_dependency_network.number_of_nodes()}")
    print(f"Edges: {workers_dependency_network.number_of_edges()}")
    print("\nTop 5 buildings by total inflow (transition counts):")
    sorted_inflow = sorted(nx.get_node_attributes(workers_dependency_network, 'inflow').items(), key=lambda item: item[1], reverse=True)
    for node, inflow in sorted_inflow[:5]:
        print(f"  Building ID: {node}, Inflow: {inflow}")
else:
    print("\nFailed to build Workers Dependency Network.")

Processing Workers...
Step 1: Spatial mapping mobility data to buildings...
  Retained mobility data after mapping to buildings: 20.40%
Step 2 & 3: Preprocessing timestamps and extracting transition sequences...
Step 4: Filtering transitions and reporting...
  Number of active buildings (nodes) with >= 20 origin visits: 113
  Number of valid cross-building transitions (within 6h & active buildings): 130777
Step 5: Constructing the dependency network...
  Network created with 113 nodes and 555 edges.
  Number of strong dependency paths (e.g., probability > 0.5): 100

Workers Dependency Network built successfully!
Nodes: 113
Edges: 555

Top 5 buildings by total inflow (transition counts):
  Building ID: db95edda-e33a-448e-975a-943d8f37c605, Inflow: 10656
  Building ID: cc72c0ea-7504-42ec-8aec-33daafcd6451, Inflow: 6225
  Building ID: 25c810fb-17d0-4331-9c9a-6cc5f5bb8042, Inflow: 6217
  Building ID: f9ed6834-375f-4fac-ae5c-d363ac9b931d, Inflow: 5323
  Building ID: e96252ed-33f8-446c-a5f0-

1.3 Visualize the actual network

In [13]:
import folium
from folium import plugins
import pandas as pd
import geopandas as gpd
import numpy as np

def visualize_functional_dependency_map(G, buildings_gdf, group_name="Workers"):
    """
    修正版：彻底解决 Building Name 为 '0' 的问题，并优化了 POI 回退显示。
    """
    # 1. 坐标转换
    nodes_4326 = buildings_gdf.to_crs(epsg=4326).set_index('id')

    # 定义功能颜色映射
    color_map = {
        'Other': '#808080', 'Small Retail/Dining': '#FFA500',
        'Shopping Mall': '#FF0000', 'Mixed-use/Complex': '#FFFF00',
        'Residential': '#008000', 'Education': '#800080',
        'Transit Hub': '#00FFFF', 'Office/HQ': '#007bff',
    }

    m = folium.Map(location=[51.5048, -0.0210], zoom_start=16, tiles='CartoDB positron')

    # 计算节点吞吐量及全局范围
    node_stats = {}
    for node_id in G.nodes():
        inf = G.nodes[node_id].get('total_inflow', 0)
        outf = G.nodes[node_id].get('total_outflow', 0)
        node_stats[str(node_id)] = {'in': inf, 'out': outf, 'total': inf + outf}

    all_totals = [s['total'] for s in node_stats.values()]
    max_tp = max(all_totals) if all_totals else 1
    min_tp = min(all_totals) if all_totals else 0

    # 2. 绘制建筑 Polygon
    for b_id, row in nodes_4326.iterrows():
        b_id_str = str(b_id)

        # --- 核心修正：多重过滤逻辑处理 Building Name ---
        raw_name = str(row.get('building_name', '')).strip()
        poi_raw = str(row.get('all_poi_names', '')).strip()
        poi_list = [p.strip() for p in poi_raw.split(',')] if poi_raw and poi_raw.lower() != 'nan' else []

        # 判断名字是否无效 (增加对 '0' 和 '0.0' 的判断)
        invalid_names = ['', 'nan', 'unknown', 'none', '0', '0.0']
        if raw_name.lower() in invalid_names:
            # 尝试回退到 POI，但要排除 "No POIs recorded"
            if poi_list and poi_list[0].lower() != "no pois recorded":
                display_name = f"{poi_list[0]} (POI)"
            else:
                display_name = f"Building {b_id_str[:8]}..." # 使用短 ID
        else:
            display_name = raw_name

        # 同步更新 G 节点中的名字，以便连线 Tooltip 显示正确
        if b_id_str in G.nodes:
            G.nodes[b_id_str]['name'] = display_name

        # 透明度计算
        stats = node_stats.get(b_id_str, {'in': 0, 'out': 0, 'total': 0})
        if max_tp > min_tp:
            norm_val = (stats['total'] - min_tp) / (max_tp - min_tp)
            node_opacity = 0.2 + (0.65 * norm_val)
        else:
            node_opacity = 0.4

        fill_color = color_map.get(row['final_functional_class'], "#95a5a6")

        # 插入 POI Num (poi_sum) 并构造内容
        raw_ratio = row.get('Retail_Food_Ratio', np.nan)
        ratio_disp = "N/A" if pd.isna(raw_ratio) else f"{raw_ratio:.1f}%"
        poi_num = int(float(row.get('poi_sum', 0))) # 确保浮点字符串也能转 int

        poi_items_html = "".join([f"<li style='margin-bottom:2px;'>{p}</li>" for p in poi_list]) if poi_list else "<li>No POIs listed</li>"

        # 构造信息框内容 (使用 popup_content)
        popup_content = f"""
        <div style="width:260px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
            <h4 style="margin:0 0 8px 0; color:#333; border-bottom:2px solid {fill_color}; padding-bottom:3px;">{display_name}</h4>
            <div style="font-size:13px; line-height:1.6;">
                <b>Functional Class:</b> {row['final_functional_class']}<br>
                <b>Throughput:</b> <span style="color:#2c3e50; font-weight:bold;">{stats['total']}</span> (In:{stats['in']}, Out:{stats['out']})<br>
                <b>POI Num:</b> {poi_num}<br>
                <b>Retail/Food Ratio:</b> {ratio_disp}
            </div>
            <details style="margin-top:10px; font-size:12px; border:1px solid #eee; border-radius:4px; padding:5px; background-color:#f9f9f9;">
                <summary style="cursor:pointer; color:#007bff; font-weight:bold;">Click to view all {poi_num} POIs</summary>
                <ul style="margin:5px 0 0 0; padding-left:18px; max-height:150px; overflow-y:auto; color:#555;">
                    {poi_items_html}
                </ul>
            </details>
        </div>
        """

        folium.GeoJson(
            row['geometry'],
            style_function=lambda x, fc=fill_color, fo=node_opacity: {
                'fillColor': fc, 'color': '#2c3e50', 'weight': 1, 'fillOpacity': fo
            },
            tooltip=f"<b>{display_name}</b><br>Throughput: {stats['total']}",
            popup=folium.Popup(popup_content, max_width=300)
        ).add_to(m)

    # 3. 叠加金色连线
    edge_layer = folium.FeatureGroup(name=f"{group_name} Network Transitions")
    all_weights = [d['weight'] for u, v, d in G.edges(data=True)]
    max_w = max(all_weights) if all_weights else 1

    for u, v, d in G.edges(data=True):
        try:
            u_s, v_s = str(u), str(v)
            p1 = nodes_4326.loc[u_s].geometry.centroid
            p2 = nodes_4326.loc[v_s].geometry.centroid
            prob = d['weight']
            line_weight = 1 + (prob / max_w * 7)

            name_u = G.nodes[u_s].get('name', u_s)
            name_v = G.nodes[v_s].get('name', v_s)

            if prob > 0.01:
                folium.PolyLine(
                    locations=[[p1.y, p1.x], [p2.y, p2.x]],
                    color="#FFD700", weight=line_weight, opacity=0.6,
                    tooltip=f"<b>Direction:</b> {name_u} &rarr; {name_v}<br><b>Prob:</b> {prob:.2%}"
                ).add_to(edge_layer)

                if prob > 0.1:
                    folium.CircleMarker(location=[p2.y, p2.x], radius=2, color='#FFD700', fill=True, fill_opacity=0.8).add_to(edge_layer)
        except Exception:
            continue

    edge_layer.add_to(m)

    # 4. 图例
    legend_html = f'''
    <div style="position: fixed; bottom: 40px; left: 40px; width: 240px; z-index:9999;
    background: white; padding: 12px; border: 2px solid grey; border-radius: 6px; font-family: Arial; font-size: 12px;">
        <b style="font-size:13px;">{group_name} Network Analysis</b><br>
        <div style="margin-top:8px; line-height:1.7;">
            <span style="color:#FFD700; font-size:18px;">━</span> <b>Dependency</b> (Thickness \u221D Prob)<br>
            <span style="color:#2c3e50; font-size:18px;">\u25A0</span> <b>Opacity</b> \u221D Throughput<br>
            <hr style="margin:8px 0; border-top:1px solid #ddd;">
            <span style="color:#007bff;">\u25A0</span> Office/HQ &nbsp; <span style="color:#FF0000;">\u25A0</span> Shopping Mall<br>
            <span style="color:#FFFF00;">\u25A0</span> Mixed-use &nbsp; <span style="color:#FFA500;">\u25A0</span> Small Retail<br>
            <span style="color:#008000;">\u25A0</span> Residential &nbsp; <span style="color:#800080;">\u25A0</span> Education<br>
            <span style="color:#00FFFF;">\u25A0</span> Transit Hub &nbsp; <span style="color:#808080;">\u25A0</span> Other
        </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl().add_to(m)
    return m

# --- 执行并保存 ---
print("🚀 Finalizing map with robust '0' name handling...")
map_final = visualize_functional_dependency_map(workers_dependency_network, buildings_gdf, "Workers")
map_final.save("Canary_Wharf_Network_Final_V3.html")
print("✅ Done! Map saved as 'Canary_Wharf_Network_Final_V3.html'")

🚀 Finalizing map with robust '0' name handling...
✅ Done! Map saved as 'Canary_Wharf_Network_Final_V3.html'


In [14]:
map_final

## **2. Simulative Counterfactual** (w.Leontief Open Model)

2.1 Remote Work -> Nearby Retail Buildings

In [15]:
import numpy as np
import pandas as pd

def simulate_refined_shock(G, buildings_gdf, shock_rate=-0.20):
    """
    改进版模拟：引入人流量权重与零售特定影响分析
    """
    # 1. 映射节点到矩阵索引
    node_list = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(node_list)}
    N = len(node_list)

    # 获取每个节点的基准流量 n (inflow)
    # 如果节点没有 inflow 属性，默认为 1 以避免除以零
    node_inflows = np.array([G.nodes[node].get('inflow', 1) for node in node_list])

    # 2. 构建依赖矩阵 W
    # W[j, i] 表示建筑 j 对建筑 i 的依赖权重 (流量 i -> j)
    W = np.zeros((N, N))
    for u, v, d in G.edges(data=True):
        if u in node_to_idx and v in node_to_idx:
            W[node_to_idx[v], node_to_idx[u]] = d['weight']

    # 3. 构建冲击向量 f (加权冲击)
    f = np.zeros(N)
    office_ids = buildings_gdf[buildings_gdf['final_functional_class'] == 'Office/HQ']['id'].astype(str).values

    found_offices = 0
    for b_id in office_ids:
        if b_id in node_to_idx:
            idx = node_to_idx[b_id]
            # 冲击 = 冲击率 * 该建筑的基准人流量
            f[idx] = shock_rate * node_inflows[idx]
            found_offices += 1

    print(f"Applying weighted shock to {found_offices} Office buildings based on their inflow size.")

    # 4. 求解 Leontief 系统：(I - W)v = f
    # 引入阻尼系数 alpha，防止数值爆炸
    # alpha 代表冲击传导的效率，建议从 0.1 开始尝试
    alpha = 0.5

    # 重新计算
    I = np.eye(N)
    # 求解 (I - alpha * W)v = f
    v_absolute = np.linalg.solve(I - alpha * W, f)

    # 这样跑出来的 total_impact_percent 应该会回归到 [-100, 0] 的正常区间

    # I = np.eye(N)
    try:
        # 求解线性方程组得到各建筑的绝对流量变化量
        v_absolute = np.linalg.solve(I - 0.99 * W, f)
    except np.linalg.LinAlgError:
        v_absolute = np.zeros(N)
        print("Error: Matrix is singular.")

    # 5. 整理并计算百分比与零售指标
    results_df = pd.DataFrame({
        'id': node_list,
        'baseline_inflow': node_inflows,
        'absolute_loss': v_absolute,
        # 计算总流量下降百分比
        'total_impact_percent': (v_absolute / node_inflows) * 100
    })

    # 合并原始建筑属性 (包括 Retail_Food_Ratio)
    final_results = results_df.merge(
        buildings_gdf[['id', 'building_name', 'final_functional_class', 'Retail_Food_Ratio', 'poi_sum']],
        on='id', how='left'
    )

    # 6. 计算 Retail-specific Impact
    # 假设 Retail_Food_Ratio 是百分数（如 50.0 代表 50%）
    # 该指标代表该建筑内零售商业部分遭受的流量打击强度
    final_results['Retail_Food_Ratio'] = pd.to_numeric(final_results['Retail_Food_Ratio'], errors='coerce').fillna(0)
    final_results['retail_specific_impact'] = (final_results['total_impact_percent'] * (final_results['Retail_Food_Ratio'] / 100))

    # 计算间接级联损失 (剔除初始冲击后的剩余损失)
    # 我们需要先把 f 转回百分比进行对比
    f_percent = (f / node_inflows) * 100
    final_results['indirect_cascade_percent'] = final_results['total_impact_percent'] - f_percent

    return final_results

# --- 运行并查看结果 ---
# 假设 workers_dependency_network 和 buildings_gdf 已在环境中
simulation_results = simulate_refined_shock(workers_dependency_network, buildings_gdf, shock_rate=-0.20)

# 过滤出非办公类建筑，观察哪些零售活跃度高的楼受灾最重
retail_analysis = simulation_results[simulation_results['total_impact_percent'] < 0].copy()
retail_analysis = retail_analysis.sort_values(by='retail_specific_impact')

print("\nTop 10 Buildings with highest Retail-specific Impact (Commercial Vulnerability):")
cols = ['building_name', 'final_functional_class', 'total_impact_percent', 'Retail_Food_Ratio', 'retail_specific_impact']
print(retail_analysis[cols].head(10))

Applying weighted shock to 43 Office buildings based on their inflow size.

Top 10 Buildings with highest Retail-specific Impact (Commercial Vulnerability):
                       building_name final_functional_class  \
65                   Crossrail Place          Shopping Mall   
25                     Credit Suisse              Office/HQ   
72                                 0          Shopping Mall   
43                         YY London              Office/HQ   
103                                0              Office/HQ   
63   Canada Square Car Park Entrance              Office/HQ   
112                                0         Infrastructure   
26                              KPMG              Office/HQ   
28                                 0              Office/HQ   
108                                0              Office/HQ   

     total_impact_percent  Retail_Food_Ratio  retail_specific_impact  
65           -2766.312113          65.116279            -1801.319516  
25     

In [16]:
import numpy as np
import pandas as pd

def simulate_percentage_shock(G, buildings_gdf, shock_rate=-0.20, alpha=0.1):
    """
    修正版：基于变化率(Percentage)的级联模拟
    f: 冲击百分比 (如 -0.2 代表 -20%)
    v = (I - alpha * W)^-1 * f
    """
    node_list = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(node_list)}
    N = len(node_list)

    # 1. 构建依赖矩阵 W
    W = np.zeros((N, N))
    for u, v, d in G.edges(data=True):
        if u in node_to_idx and v in node_to_idx:
            # W[v, u] 代表 v 依赖于 u
            W[node_to_idx[v], node_to_idx[u]] = d['weight']

    # 2. 构建冲击向量 f (直接使用百分比)
    f = np.zeros(N)
    office_ids = buildings_gdf[buildings_gdf['final_functional_class'] == 'Office/HQ']['id'].astype(str).values

    for b_id in office_ids:
        if b_id in node_to_idx:
            f[node_to_idx[b_id]] = shock_rate

    # 3. 求解线性方程组 (I - alpha * W)v = f
    # alpha 是传导率，beta_w 在论文中通常显著提升预测力 [cite: 287, 402]
    I = np.eye(N)
    try:
        v_percent = np.linalg.solve(I - alpha * W, f)
    except np.linalg.LinAlgError:
        v_percent = np.linalg.solve(I - (alpha * 0.9) * W, f)

    # 4. 整理结果
    results_df = pd.DataFrame({
        'id': node_list,
        'total_impact_percent': v_percent * 100, # 转回 0-100 格式
        'initial_shock_percent': f * 100
    })

    final_results = results_df.merge(
        buildings_gdf[['id', 'building_name', 'final_functional_class', 'Retail_Food_Ratio']],
        on='id', how='left'
    )

    # 5. 计算 Retail-specific Impact
    final_results['Retail_Food_Ratio'] = pd.to_numeric(final_results['Retail_Food_Ratio'], errors='coerce').fillna(0)
    final_results['retail_specific_impact'] = (final_results['total_impact_percent'] * (final_results['Retail_Food_Ratio'] / 100))

    return final_results

# 建议先从 alpha = 0.5 开始尝试
simulation_results = simulate_percentage_shock(workers_dependency_network, buildings_gdf, shock_rate=-0.20, alpha=0.5)

print(simulation_results[simulation_results['initial_shock_percent'] == 0].sort_values(by='total_impact_percent').head(10))

                                       id  total_impact_percent  \
67   3a0a73a0-b21b-4536-9fe0-49ac6cf926f9            -10.071430   
13   fe48f30d-5eb3-4ece-94f6-be71071b58ca             -9.569802   
45   e089f4d0-5512-4aa2-b973-782c910fbd62             -8.141345   
18   0142a0c6-a403-459a-87ab-20655f0281c5             -6.920722   
40   9dc3ac14-3476-4029-980c-b21665062ed2             -6.528993   
24   44001ba7-9ea1-45b2-bc34-daeba4ac0440             -3.691068   
0    db95edda-e33a-448e-975a-943d8f37c605             -2.675617   
62   5df0e794-2aeb-4f9f-970b-5f9c3bb14fac             -2.453860   
65   7d04c4cc-c183-4597-bb98-063679b2ec63             -1.930536   
112  e98f7d70-3433-4f7a-a661-04fc7fd887fa             -1.630617   

     initial_shock_percent                   building_name  \
67                     0.0       Vertus - 10 George Street   
13                     0.0  Discovery Dock Apartments East   
45                     0.0                               0   
18            

In [17]:
import folium
from folium import plugins
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

def visualize_shock_impact_map(G, buildings_gdf, simulation_results, group_name="Workers"):
    """
    基于反事实模拟结果的冲击可视化地图
    颜色深浅表示流量损失程度 (total_impact_percent)
    """
    # 1. 数据合并与坐标转换
    # 确保 simulation_results 的 id 是字符串类型以匹配索引
    simulation_results['id'] = simulation_results['id'].astype(str)
    impact_gdf = buildings_gdf.merge(
        simulation_results[['id', 'total_impact_percent', 'retail_specific_impact', 'initial_shock_percent']],
        on='id',
        how='inner'
    )
    nodes_4326 = impact_gdf.to_crs(epsg=4326).set_index('id')

    # 2. 设置颜色映射 (Colormap)
    # 使用红色渐变，损失越重（负值越大）颜色越深
    min_impact = nodes_4326['total_impact_percent'].min()
    # 归一化：将最小负值映射到 1.0 (深红)，0 映射到 0.0 (浅色)
    norm = colors.Normalize(vmin=min_impact, vmax=0)
    impact_cmap = plt.get_cmap('Reds_r')

    m = folium.Map(location=[51.5048, -0.0210], zoom_start=16, tiles='CartoDB dark_matter')

    # 3. 绘制建筑 Polygon
    for b_id, row in nodes_4326.iterrows():
        b_id_str = str(b_id)
        impact_val = row['total_impact_percent']
        retail_impact = row['retail_specific_impact']
        is_shock_source = row['initial_shock_percent'] < 0

        # 计算填充颜色
        if impact_val < 0:
            fill_color = colors.to_hex(impact_cmap(norm(impact_val)))
        else:
            fill_color = "#34495e" # 无显著变化

        # 处理 Building Name
        raw_name = str(row.get('building_name', '')).strip()
        display_name = raw_name if raw_name.lower() not in ['', 'nan', '0', '0.0'] else f"Building {b_id_str[:8]}"

        # 构造信息框
        source_tag = "<b style='color:#e74c3c;'>[Shock Source: Office]</b><br>" if is_shock_source else ""
        popup_content = f"""
        <div style="width:240px; font-family: sans-serif;">
            <h4 style="margin:0 0 5px 0;">{display_name}</h4>
            {source_tag}
            <b>Class:</b> {row['final_functional_class']}<br>
            <hr style='margin:5px 0;'>
            <div style="font-size:13px;">
                <b style="color:#e74c3c;">Total Flow Impact: {impact_val:.2f}%</b><br>
                <b style="color:#3498db;">Retail-Specific Loss: {retail_impact:.2f}</b><br>
                <small>(Retail Ratio: {row.get('Retail_Food_Ratio', 0):.1f}%)</small>
            </div>
        </div>
        """

        folium.GeoJson(
            row['geometry'],
            style_function=lambda x, fc=fill_color, source=is_shock_source: {
                'fillColor': fc,
                'color': '#ecf0f1' if source else '#95a5a6',
                'weight': 2 if source else 0.5,
                'fillOpacity': 0.8
            },
            tooltip=f"<b>{display_name}</b><br>Impact: {impact_val:.2f}%",
            popup=folium.Popup(popup_content, max_width=300)
        ).add_to(m)

    # 4. 叠加“级联路径” (Golden Paths)
    # 只显示指向受灾最严重建筑的强依赖连线，揭示冲击传播路径
    edge_layer = folium.FeatureGroup(name="Cascading Impact Paths")
    # 筛选出受灾最重的 Top 15 建筑
    top_victims = simulation_results.sort_values(by='total_impact_percent').head(15)['id'].values

    max_w = max([d['weight'] for u, v, d in G.edges(data=True)]) if G.number_of_edges() > 0 else 1

    for u, v, d in G.edges(data=True):
        u_s, v_s = str(u), str(v)
        # 如果目的地 v 受灾严重，且 u 是 Office (Shock Source)
        if v_s in top_victims and u_s in nodes_4326.index:
            try:
                p1 = nodes_4326.loc[u_s].geometry.centroid
                p2 = nodes_4326.loc[v_s].geometry.centroid
                prob = d['weight']

                # 只有依赖概率足够大时才绘制
                if prob > 0.05:
                    folium.PolyLine(
                        locations=[[p1.y, p1.x], [p2.y, p2.x]],
                        color="#f1c40f", # 金色连线
                        weight=1 + (prob/max_w * 5),
                        opacity=0.6,
                        tooltip=f"Dependency Strength: {prob:.2%}"
                    ).add_to(edge_layer)
            except:
                continue

    edge_layer.add_to(m)

    # 5. 图例
    legend_html = f'''
    <div style="position: fixed; bottom: 40px; left: 40px; width: 220px; z-index:9999;
    background: rgba(255,255,255,0.9); padding: 12px; border-radius: 8px; font-family: Arial; font-size: 12px; border:1px solid #ccc;">
        <b style="font-size:13px;">Counterfactual Simulation</b><br>
        <small>Scenario: -20% Remote Work Shock</small><br>
        <hr style="margin:8px 0;">
        <div style="margin-top:5px;">
            <span style="background:linear-gradient(to right, #fee0d2, #de2d26); width:100%; height:10px; display:block;"></span>
            <div style="display:flex; justify-content:space-between; margin-bottom:10px;">
                <span>0% Impact</span><span>Max Loss</span>
            </div>
            <span style="color:#f1c40f; font-size:18px;">━</span> <b>Cascading Path</b> (w_ij)<br>
            <span style="border:2px solid #ecf0f1; width:12px; height:12px; display:inline-block;"></span> <b>Shock Source</b> (Office)
        </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl().add_to(m)

    return m

# --- 执行并保存 ---
print("🚀 Generating Shock Impact Map...")
# 使用之前跑出的 simulation_results 和 alpha 修正后的逻辑
impact_map_final = visualize_shock_impact_map(workers_dependency_network, buildings_gdf, simulation_results, "Workers")
impact_map_final.save("Canary_Wharf_Shock_Impact_V1.html")
print("✅ Done! Map saved as 'Canary_Wharf_Shock_Impact_V1.html'")

🚀 Generating Shock Impact Map...
✅ Done! Map saved as 'Canary_Wharf_Shock_Impact_V1.html'


In [18]:
impact_map_final

## **3. Data Output**

In [19]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString

# --- 1. Export Buildings with Simulation and Network Node Attributes ---
print("🚀 Preparing buildings_with_impact.geojson...")

# Get node attributes from workers_dependency_network
node_attributes = []
for node_id, data in workers_dependency_network.nodes(data=True):
    node_attributes.append({
        'id': node_id,
        'inflow': data.get('inflow', 0),
        'outflow': data.get('outflow', 0)
    })
network_node_attrs_df = pd.DataFrame(node_attributes)

# Merge simulation results and network node attributes with buildings_gdf
buildings_export_gdf = buildings_gdf.merge(simulation_results,
                                          on='id', how='left')
buildings_export_gdf = buildings_export_gdf.merge(network_node_attrs_df,
                                                  on='id', how='left')

# Fill NaN values for buildings not in the network or simulation results
buildings_export_gdf['total_impact_percent'] = buildings_export_gdf['total_impact_percent'].fillna(0)
buildings_export_gdf['retail_specific_impact'] = buildings_export_gdf['retail_specific_impact'].fillna(0)
buildings_export_gdf['initial_shock_percent'] = buildings_export_gdf['initial_shock_percent'].fillna(0)
buildings_export_gdf['inflow'] = buildings_export_gdf['inflow'].fillna(0)
buildings_export_gdf['outflow'] = buildings_export_gdf['outflow'].fillna(0)

# Ensure 'id' column is string for consistency if not already
buildings_export_gdf['id'] = buildings_export_gdf['id'].astype(str)

# Save to GeoJSON
buildings_export_gdf.to_file("buildings_with_impact.geojson", driver='GeoJSON')
print("✅ 'buildings_with_impact.geojson' saved successfully!")

# --- 2. Export Dependency Network Edges ---
print("🚀 Preparing dependency_edges.geojson...")

edges_data = []
# Reproject buildings_gdf to EPSG:4326 for centroid extraction if not already
buildings_4326 = buildings_gdf.to_crs(epsg=4326).set_index('id')

for u, v, data in workers_dependency_network.edges(data=True):
    try:
        # Get centroids for the source (u) and target (v) nodes
        centroid_u = buildings_4326.loc[str(u)].geometry.centroid
        centroid_v = buildings_4326.loc[str(v)].geometry.centroid

        # Create a LineString geometry
        line_geometry = LineString([(centroid_u.x, centroid_u.y), (centroid_v.x, centroid_v.y)])

        edges_data.append({
            'source_building_id': str(u),
            'target_building_id': str(v),
            'weight': data.get('weight', 0.0),
            'transitions': data.get('transitions', 0),
            'geometry': line_geometry
        })
    except KeyError: # Handle cases where building ID might not be in buildings_gdf
        continue

if edges_data:
    edges_gdf = gpd.GeoDataFrame(edges_data, crs="EPSG:4326")
    edges_gdf.to_file("dependency_edges.geojson", driver='GeoJSON')
    print("✅ 'dependency_edges.geojson' saved successfully!")
else:
    print("ℹ️ No valid edges to export for dependency_edges.geojson.")


🚀 Preparing buildings_with_impact.geojson...
✅ 'buildings_with_impact.geojson' saved successfully!
🚀 Preparing dependency_edges.geojson...
✅ 'dependency_edges.geojson' saved successfully!


In [20]:
import geopandas as gpd
import pandas as pd

print("🚀 Preparing to export data to GeoPackage...")

# --- 1. Buildings Layer (from buildings_export_gdf) ---
# buildings_export_gdf is already prepared from the previous steps
# It contains buildings_gdf merged with simulation_results and network_node_attrs_df

# --- 2. Edges Layer (from edges_gdf) ---
# edges_gdf is already prepared from the previous steps
# It contains the LineString geometries for dependencies

# Define the output GeoPackage file name
gpkg_output_path = "Canary_Wharf_Network_And_Impact.gpkg"

# Export the buildings_export_gdf as the 'buildings' layer
buildings_export_gdf.to_file(gpkg_output_path, layer='buildings', driver="GPKG")
print("✅ Buildings data exported to GeoPackage layer 'buildings'.")

# Export the edges_gdf as the 'edges' layer to the SAME GPKG file
edges_gdf.to_file(gpkg_output_path, layer='edges', driver="GPKG", mode='a') # 'a' for append mode
print("✅ Dependency edges data exported to GeoPackage layer 'edges'.")

print("🎉 All data successfully exported to 'Canary_Wharf_Network_And_Impact.gpkg'!")

🚀 Preparing to export data to GeoPackage...
✅ Buildings data exported to GeoPackage layer 'buildings'.
✅ Dependency edges data exported to GeoPackage layer 'edges'.
🎉 All data successfully exported to 'Canary_Wharf_Network_And_Impact.gpkg'!
